# MESSAGEix-Pakistan 
### Baseline Model
In this notebook, we are reading data and building baseline scenerio.

<img src="https://wit.lums.edu.pk/sites/default/files/inline-images/WIT_Banner.jpg" alt="Girl in a jacket" width="850" height="250">

In [1]:
# Autoreload modules when changes are applied to them
%load_ext autoreload
%autoreload all
%matplotlib inline


In [2]:
# fundamental libraries
import os
import sys
from pathlib import Path

# Resolve the message_pak root (one level above this notebook's directory)
ROOT = Path(os.path.abspath('')).parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import ixmp
import message_ix
from message_ix import log
%matplotlib inline


In [3]:
# Confirm root path (message_pak directory)
print('Project root:', ROOT)


Project root: /Users/awais/Documents/GitHub/NEST-Pakistan/messageix_PAK


Create scenario

In [4]:
# creating ixmp platform object
new_mp = ixmp.Platform()

# creating a new, empty scenario object
scenario = message_ix.Scenario(
    new_mp, model="MESSAGEix-Pakistan", scenario="baseline", version="new"
)

2026-03-17 11:35:54,246  INFO at.ac.iiasa.ixmp.Platform:165 - Welcome to the IX modeling platform!
2026-03-17 11:35:54,250  INFO at.ac.iiasa.ixmp.Platform:166 -  connected to database 'jdbc:hsqldb:file:/Users/awais/.local/share/ixmp/localdb/default' (user: ixmp)...


Read Data

In [6]:
data_path = ROOT / "data" / "modelfiles" / "MESSAGEix-Pakistan-CurrentMeasures-SSP2.xlsx"
scenario.read_excel(str(data_path), add_units=True, commit_steps=False, init_items=True)

##### Solve the Model

In [94]:
log.info(f"version number before commit(): {scenario.version}")


# exporting the built model (Scenario) to GAMS with an optional case name
caseName = scenario.model + '__' + scenario.scenario + '__v' + str(scenario.version)

# solve model
scenario.solve(case=caseName)


--- Warning: The GAMS version [49.3.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 03/17/26 12:59:11 49.3.0 7de46a92 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/49/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/awais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/awais/Documents/GitHub/message_ix/message_ix/model/225b/
    SysDir /Library/Frameworks/GAMS.framework/Versions/49/Resources/
    LogOption 4
    --in /Users/awais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-Pakistan__baseline__v4.gdx
    --out /Users/awais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-Pakistan__baseline__v4.gdx
    --iter /Users/awais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-Pakistan__baseline__v4.gdx
Licensee: Small MUD - 5 User License                     S240905|0002AP-GEN
          IIASA, Informatio

2026-03-17 12:59:15,635 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2026-03-17 12:59:15,636 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [102]:
scenario.remove_solution()

In [115]:
df = scenario.set("map_shares_commodity_share")

In [117]:
scenario.to_excel('model_file.xlsx')

In [ ]:
df['shares'] = 'UE_transport_elec'
df["value"] = 0.7

In [110]:
with scenario.transact(""):
    # scenario.add_set("shares", "UE_transport_elec")
    scenario.remove_par("share_commodity_up", df)

In [91]:
df = scenario.par('bound_activity_up', filters = {'technology':'elec_trp'})

In [92]:
df['value'] = df['value'] /0.33

In [93]:
with scenario.transact(''):
    scenario.add_par('bound_activity_up', df)

In [76]:
new_years_values = {
    2035: 1.843219588,
    2040: 5.774046306,
    2045: 12.30737802,
    2050: 20.45385382,
    2055: 29.06134165,
    2060: 37.4789135,
    2070: 53.22169814,
}

template = df.iloc[0].to_dict()
new_rows = []
for year, value in new_years_values.items():
    row = template.copy()
    row["year_act"] = year
    row["value"] = value
    new_rows.append(row)

df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)


In [80]:
with scenario.transact(''):
    scenario.add_par('bound_activity_lo', df)

##### Reporting Results

In [95]:
from report.legacy.iamc_report_hackathon import report
from datetime import datetime
import time

timestamp = datetime.now().strftime('%Y-%m-%d--%H-%M')
start = time.time()
out_dir = ROOT / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
df, path_name = report(mp=new_mp, scen=scenario, out_dir=str(out_dir), out_file_timestamp=timestamp, IDEA_format=False)
end = time.time()
print(f'Reporting done in {end - start:.1f}s → {out_dir}')


processing Table: Resource|Extraction
processing Table: Resource|Cumulative Extraction
processing Table: Primary Energy
processing Table: Primary Energy (substitution method)
processing Table: Final Energy
processing Table: Secondary Energy|Electricity
processing Table: Secondary Energy|Heat
processing Table: Secondary Energy
processing Table: Secondary Energy|Gases
processing Table: Secondary Energy|Solids
processing Table: Emissions|CO2
processing Table: Carbon Sequestration
processing Table: Emissions|BC
processing Table: Emissions|OC
processing Table: Emissions|CO
processing Table: Emissions|N2O
processing Table: Emissions|CH4
processing Table: Emissions|NH3
processing Table: Emissions|Sulfur
processing Table: Emissions|NOx
processing Table: Emissions|VOC
processing Table: Emissions|HFC
HFC125
No objects to concatenate
processing Table: Emissions
processing Table: Emissions
processing Table: Agricultural Demand
module 'report.legacy.default_tables_static' has no attribute 'retr_agr

/Users/awais/Documents/GitHub/message-ix-models/message_ix_models/util/compat/message_data/utilities.py:191: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.


no emissions included


/Users/awais/Documents/GitHub/message-ix-models/message_ix_models/util/compat/message_data/utilities.py:191: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
/Users/awais/Documents/GitHub/message-ix-models/message_ix_models/util/compat/message_data/utilities.py:191: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
/Users/awais/Documents/GitHub/message-ix-models/message_ix_models/util/compat/message_data/utilities.py:191: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
/Users/awais/Documents/GitHub/message-ix-models/message_ix_models/util/compat/message_data/utilities.py:191: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
/Users/awais/Documents/GitHub/message-ix-models/message_ix_models/util/compat/message_data/utilities.py:191: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
/Users/awais/Documents/GitHub/message-ix-mode

no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
no emissions included
Reporting done in 208.3s → /Users/awais/Documents/GitHub/NEST-Pakistan/messageix_PAK/output


In [ ]:
# close the connection to the database
new_mp.close_db() 